# 01 - Exploration des données

Objectif : comprendre les données avant d'entraîner quoi que ce soit.

1. Charger toutes les sources dans un seul tableau
2. Vérifier la qualité (valeurs manquantes, textes vides, doublons)
3. Regarder l'équilibre attaques / messages normaux
4. Détecter les langues
5. Étudier la longueur des textes
6. Lire des exemples au hasard
7. Sauvegarder un fichier propre et unifié pour la suite

Ce notebook se lance depuis la racine du projet ou depuis le dossier `notebooks/`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

# Trouve la racine du projet, que le notebook soit lancé depuis la racine ou depuis notebooks/
RACINE = Path.cwd()
if not (RACINE / "data").exists():
    RACINE = RACINE.parent
DATA = RACINE / "data"
print("Racine du projet :", RACINE)

## 1. Chargement de toutes les sources

In [ ]:
def lire_jsonl(chemin):
    with open(chemin, encoding="utf-8") as f:
        return [json.loads(ligne) for ligne in f if ligne.strip()]

fichiers = sorted((DATA / "raw").glob("*.jsonl")) + sorted((DATA / "fr").glob("*.jsonl"))
lignes = []
for chemin in fichiers:
    contenu = lire_jsonl(chemin)
    for l in contenu:
        l.setdefault("source", f"gandalf-le-gris:{chemin.stem}")
        l["fichier"] = chemin.stem
    lignes.extend(contenu)
    print(f"{chemin.relative_to(RACINE)} : {len(contenu)} exemples")

df = pd.DataFrame(lignes)
print("\nTotal :", len(df), "exemples,", df.shape[1], "colonnes")
df.head()

## 2. Qualité des données

In [ ]:
df["texte"] = df["texte"].astype(str)
qualite = pd.DataFrame({
    "valeurs_manquantes": df.isna().sum(),
    "valeurs_distinctes": df.nunique(),
})
print(qualite)

vides = (df["texte"].str.strip() == "").sum()
espaces = (df["texte"] != df["texte"].str.strip()).sum()
print("\nTextes vides :", vides)
print("Textes avec espaces au début ou à la fin :", espaces)
print("Identifiants en double :", df["id"].duplicated().sum())

### Doublons de texte

Un même texte présent deux fois est dangereux : s'il se retrouve à la fois dans l'entraînement
et dans le test, les scores seront faussement bons. Un texte présent avec deux labels différents
est encore pire, c'est une contradiction dans les données.

In [ ]:
df["texte_norm"] = df["texte"].str.lower().str.split().str.join(" ")

doublons = df[df.duplicated("texte_norm", keep=False)].sort_values("texte_norm")
print("Lignes concernées par un doublon :", len(doublons))

conflits = df.groupby("texte_norm")["label"].nunique()
conflits = conflits[conflits > 1]
print("Textes avec des labels contradictoires :", len(conflits))

doublons[["fichier", "label", "texte"]].head(20)

## 3. Équilibre des classes

In [ ]:
print(df["label"].value_counts().rename({1: "attaque", 0: "normal"}))
print()
tableau = pd.crosstab(df["fichier"], df["label"].map({1: "attaque", 0: "normal"}), margins=True)
tableau

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["label"].map({1: "attaque", 0: "normal"}).value_counts().plot.bar(ax=axes[0], title="Attaques et messages normaux")
df["categorie"].value_counts().plot.barh(ax=axes[1], title="Répartition par catégorie")
plt.tight_layout()
plt.show()

**À noter dans ton journal de bord :** si les attaques sont beaucoup plus nombreuses que les messages
normaux, le modèle risque d'apprendre à tout bloquer. Il faudra rééquilibrer, surtout en ajoutant
des messages normaux et des faux amis, ou utiliser des poids de classes.

## 4. Détection des langues

Les datasets publics n'indiquent pas toujours la langue (deepset mélange anglais et allemand).
On la détecte automatiquement avec `langdetect`. Ce n'est pas parfait sur les textes très courts,
mais c'est suffisant pour avoir une vue d'ensemble.

In [ ]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # rend la détection reproductible

def detecter_langue(texte):
    try:
        return detect(texte)
    except Exception:
        return "inconnue"

df["langue_detectee"] = df["texte"].apply(detecter_langue)
pd.crosstab(df["langue_detectee"], df["label"].map({1: "attaque", 0: "normal"})).sort_values("attaque", ascending=False).head(10)

In [ ]:
# Pour les exemples dont la langue n'était pas renseignée, on garde la langue détectée
masque = df["langue"].isin(["inconnue"]) | df["langue"].isna()
df.loc[masque, "langue"] = df.loc[masque, "langue_detectee"]
df["langue"].value_counts().head(10)

## 5. Longueur des textes

In [ ]:
df["nb_caracteres"] = df["texte"].str.len()
df["nb_mots"] = df["texte"].str.split().str.len()

print(df.groupby(df["label"].map({1: "attaque", 0: "normal"}))["nb_mots"].describe(percentiles=[.5, .95, .99]))

fig, ax = plt.subplots(figsize=(10, 4))
for label, nom in [(0, "normal"), (1, "attaque")]:
    df.loc[df["label"] == label, "nb_mots"].clip(upper=300).plot.hist(bins=50, alpha=0.6, label=nom, ax=ax)
ax.set_xlabel("Nombre de mots (plafonné à 300)")
ax.set_title("Longueur des textes")
ax.legend()
plt.show()

**Deux questions à se poser ici :**

- Si les attaques sont systématiquement plus longues que les messages normaux, le modèle peut
  tricher en regardant la longueur au lieu du sens. Il faudra ajouter des attaques courtes et des
  messages normaux longs.
- Les transformers ont une limite de longueur (souvent 512 tokens). Le percentile 99 te dit si
  beaucoup de textes la dépassent et s'il faudra les découper.

## 6. Lecture d'exemples

In [ ]:
# Relance cette cellule plusieurs fois : lire les données à la main est la meilleure façon d'y repérer des erreurs
for label, nom in [(1, "ATTAQUES"), (0, "NORMAUX")]:
    print(f"===== {nom} =====")
    for _, l in df[df["label"] == label].sample(5).iterrows():
        print(f"[{l['fichier']} | {l['categorie']}] {l['texte'][:200]}")
    print()

## 7. Sauvegarde du dataset unifié

On supprime les textes vides et les doublons exacts (en gardant la première occurrence),
on écarte les textes aux labels contradictoires pour les vérifier à la main, puis on sauvegarde.

In [ ]:
propre = df[df["texte"].str.strip() != ""].copy()
propre = propre[~propre["texte_norm"].isin(conflits.index)]
avant = len(propre)
propre = propre.drop_duplicates("texte_norm", keep="first")
print("Doublons supprimés :", avant - len(propre))

colonnes = ["id", "texte", "label", "categorie", "type", "langue", "groupe", "source"]
sortie = DATA / "processed"
sortie.mkdir(parents=True, exist_ok=True)

propre[colonnes].to_json(sortie / "dataset_unifie.jsonl", orient="records", lines=True, force_ascii=False)
df[df["texte_norm"].isin(conflits.index)][colonnes].to_json(sortie / "a_verifier.jsonl", orient="records", lines=True, force_ascii=False)

print("Exemples sauvegardés :", len(propre))
print(propre["label"].map({1: "attaque", 0: "normal"}).value_counts())

## Ce qu'il faut retenir

Note ici, avec tes propres mots, les chiffres clés que tu as observés. Ils serviront directement
pour la section « Données » du README :

- Nombre total d'exemples après nettoyage :
- Proportion attaques / normaux :
- Part du français :
- Problèmes repérés et décisions prises :